In [1]:
import ModelCatalog
from utils import model_utils
from dataset.DataLoader import create_dataloaders
from FineTuner import fine_tune
from FineTuneStrategy import FineTuneStrategy

C:\Users\sinat\anaconda3\envs\breast-cancer-agent\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\sinat\anaconda3\envs\breast-cancer-agent\lib\site-packages\torch\utils\cpp_extension.py:28: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging  # type: ignore[attr-defined]


## Default model catalog
For the default catalog I have provided the agent with 5 models. 2 of these are vision transformers and the rest are classical deep convolutional nets.

In [2]:
catalog = ModelCatalog.get_default_catalog()
print(catalog)

Models:
Model name: vit_b_16, accuracy: 81.072, params: 86.6, gflops: 17.56
Model name: swin_s, accuracy: 83.196, params: 49.6, gflops: 8.74
Model name: resnet50, accuracy: 76.13, params: 25.6, gflops: 4.09
Model name: mobilenet_v3_large, accuracy: 74.042, params: 5.5, gflops: 0.22
Model name: densenet161, accuracy: 77.138, params: 28.7, gflops: 7.73



As you can see in the above cell, the param count ranges from 5.5M to 86.6M. The goal was to provide the model with different tools, each capable of achieving differing levels of accuracy with different parameter counts.

In [3]:
models_data = [
    catalog.get_model_data("vit_b_16"),
    catalog.get_model_data("swin_s"),
    catalog.get_model_data("resnet50"),
    catalog.get_model_data("densenet161"),
    catalog.get_model_data("mobilenet_v3_large")
]
models = [model_utils.get_model(model) for model in models_data]
print(models[0])

BreastCancerModel(
  (base_model): VisionTransformer(
    (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (encoder): Encoder(
      (dropout): Dropout(p=0.0, inplace=False)
      (layers): Sequential(
        (encoder_layer_0): EncoderBlock(
          (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (self_attention): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (dropout): Dropout(p=0.0, inplace=False)
          (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): MLPBlock(
            (0): Linear(in_features=768, out_features=3072, bias=True)
            (1): GELU(approximate='none')
            (2): Dropout(p=0.0, inplace=False)
            (3): Linear(in_features=3072, out_features=768, bias=True)
            (4): Dropout(p=0.0, inplace=False)
          )
        )
        (encoder_layer_1): EncoderBlock(
          

In [4]:
train_loader, val_loader, test_loader = create_dataloaders(batch_size=8)

Created Dataframes (800, 2), (100, 2), (100, 2)


In [5]:
vit_lora = fine_tune(models[0],
                FineTuneStrategy.LORA,
                train_loader=train_loader,
                val_loader=val_loader,
                num_epochs=15,
                )
vit_lora

trainable params: 1,769,472 || all params: 87,571,973 || trainable%: 2.0206
class_weights: tensor([0.1548, 0.4595, 1.3293, 1.2407, 1.8157], device='cuda:0')

Starting fine-tuning with strategy: LORA
Device: cuda, Epochs: 15, LR: 0.0001
--------------------------------------------------------------------------------
Epoch 1/15
  Train Loss: 1.6585
  Val Loss: 1.5459
  Val Accuracy: 0.5000
  Val F1 (Weighted): 0.5153
  Val F1 (Macro): 0.3092
  Val Precision: 0.3318
  Val Recall: 0.3362
--------------------------------------------------------------------------------
Epoch 2/15
  Train Loss: 1.5553
  Val Loss: 1.5274
  Val Accuracy: 0.6400
  Val F1 (Weighted): 0.5057
  Val F1 (Macro): 0.1580
  Val Precision: 0.1306
  Val Recall: 0.2000
--------------------------------------------------------------------------------
Epoch 3/15
  Train Loss: 1.4812
  Val Loss: 1.5033
  Val Accuracy: 0.5000
  Val F1 (Weighted): 0.5214
  Val F1 (Macro): 0.3033
  Val Precision: 0.2900
  Val Recall: 0.3808
-----

{'train_losses': [1.6585359644889832,
  1.555333526134491,
  1.4811771667003633,
  1.3781042832136154,
  1.1883056253194808,
  1.0651125997304915,
  0.9700799679756165,
  0.8104287305474281,
  0.695444047152996,
  0.5711081287264824,
  0.4652828462421894,
  0.40814937770366666,
  0.37620606817305086,
  0.33901410549879074,
  0.34199642710387707],
 'val_losses': [1.5458719913776104,
  1.5273613379551814,
  1.50333833694458,
  1.5368768068460317,
  1.5409027292178228,
  1.6329458218354445,
  1.7159341940513024,
  1.7761655770815337,
  1.8130147548822255,
  1.8890177882634676,
  1.9577944507965674,
  2.0332487798654117,
  2.0659497792904196,
  2.072144948519193,
  2.0754941885287943],
 'val_accuracies': [0.5,
  0.64,
  0.5,
  0.46,
  0.54,
  0.61,
  0.44,
  0.35,
  0.47,
  0.32,
  0.55,
  0.52,
  0.51,
  0.54,
  0.55],
 'val_f1_scores': [0.5,
  0.64,
  0.5,
  0.46,
  0.54,
  0.61,
  0.44,
  0.35,
  0.47,
  0.32,
  0.55,
  0.52,
  0.51,
  0.54,
  0.55],
 'val_precision_scores': [0.33184496

In [6]:
resnet_full_ft = fine_tune(models[2],
                          strategy=FineTuneStrategy.FULL_FT,
                          train_loader=train_loader,
                          val_loader=val_loader)
resnet_full_ft

class_weights: tensor([0.1548, 0.4595, 1.3293, 1.2407, 1.8157], device='cuda:0')

Starting fine-tuning with strategy: FULL_FT
Device: cuda, Epochs: 10, LR: 0.0001
--------------------------------------------------------------------------------
Epoch 1/10
  Train Loss: 1.6117
  Val Loss: 1.6329
  Val Accuracy: 0.1600
  Val F1 (Weighted): 0.0736
  Val F1 (Macro): 0.0848
  Val Precision: 0.0546
  Val Recall: 0.1929
--------------------------------------------------------------------------------
Epoch 2/10
  Train Loss: 1.3681
  Val Loss: 1.7783
  Val Accuracy: 0.3300
  Val F1 (Weighted): 0.3852
  Val F1 (Macro): 0.2315
  Val Precision: 0.2719
  Val Recall: 0.3212
--------------------------------------------------------------------------------
Epoch 3/10
  Train Loss: 1.1017
  Val Loss: 2.0469
  Val Accuracy: 0.2500
  Val F1 (Weighted): 0.2794
  Val F1 (Macro): 0.2571
  Val Precision: 0.3162
  Val Recall: 0.3835
------------------------------------------------------------------------------

{'train_losses': [1.6117170858383179,
  1.3681009870767593,
  1.1017049691081047,
  0.5828547267615796,
  0.34000934690237045,
  0.17563694793730975,
  0.09247008297592402,
  0.05743638168554753,
  0.0394674818823114,
  0.038965966715477404],
 'val_losses': [1.6329091833187983,
  1.7782789377065806,
  2.046862473854652,
  2.2796877806003275,
  2.4944068422684302,
  2.900629107768719,
  2.77161266712042,
  2.656432628631592,
  2.7342836214945865,
  2.9078906636971693],
 'val_accuracies': [0.16, 0.33, 0.25, 0.65, 0.57, 0.65, 0.62, 0.6, 0.58, 0.62],
 'val_f1_scores': [0.16, 0.33, 0.25, 0.65, 0.57, 0.65, 0.62, 0.6, 0.58, 0.62],
 'val_precision_scores': [0.05458937198067633,
  0.2719411127769448,
  0.31619047619047613,
  0.3096969696969697,
  0.33961788031723145,
  0.32842377260981914,
  0.28967320261437907,
  0.30337552742616036,
  0.46641414141414145,
  0.3007575757575757],
 'val_recall_scores': [0.19285714285714287,
  0.3211805555555555,
  0.38353174603174606,
  0.2755952380952381,
  0.3

In [5]:
mobilenet_probe = fine_tune(models[4],
                            strategy=FineTuneStrategy.LINEAR_PROBE,
                            train_loader=train_loader,
                            val_loader=val_loader)
mobilenet_probe

class_weights: tensor([0.1548, 0.4595, 1.3293, 1.2407, 1.8157], device='cuda:0')

Starting fine-tuning with strategy: LINEAR_PROBE
Device: cuda, Epochs: 10, LR: 0.0001
--------------------------------------------------------------------------------
Epoch 1/10
  Train Loss: 1.5962
  Val Loss: 1.6422
  Val Accuracy: 0.5500
  Val F1 (Weighted): 0.4816
  Val F1 (Macro): 0.1673
  Val Precision: 0.1611
  Val Recall: 0.1847
--------------------------------------------------------------------------------
Epoch 2/10
  Train Loss: 1.5480
  Val Loss: 1.6248
  Val Accuracy: 0.5700
  Val F1 (Weighted): 0.5221
  Val F1 (Macro): 0.1978
  Val Precision: 0.2109
  Val Recall: 0.2037
--------------------------------------------------------------------------------
Epoch 3/10
  Train Loss: 1.4948
  Val Loss: 1.6281
  Val Accuracy: 0.5400
  Val F1 (Weighted): 0.4807
  Val F1 (Macro): 0.1834
  Val Precision: 0.1833
  Val Recall: 0.2720
-------------------------------------------------------------------------

FineTuneResult(model_name='mobilenet_v3_large', fine_tune_strategy='LINEAR_PROBE', model_dir='../model_dir/mobilenet_v3_large-LINEAR_PROBE', loss=1.5862101774949293, accuracy=0.44, precision=0.2614420667052246, recall=0.2790674603174603, f1_micro=0.44, f1_macro=0.22151067323481116, f1_weighted=0.47621510673234807, confusion_matrix=[[38, 12, 1, 2, 11], [8, 4, 1, 2, 6], [3, 4, 1, 1, 0], [2, 1, 0, 0, 1], [1, 0, 0, 0, 1]])